# Stage Silver — WHO Mortalidad

Transforma las 2 tablas Bronze de WHO hacia `stage_silver_ss2`.

| Tabla Bronze | Tabla Silver |
|---|---|
| `who_guatemala` | `who_guatemala` |
| `who_costa_rica` | `who_costa_rica` |

**Mapeo Bronze → Silver (nombres descriptivos en español, consistente con INE/MSPAS):**

| Columna Bronze | Columna Silver | Notas |
|---|---|---|
| `indicator_code` | `codigo_causa` | |
| `indicator_name` | `nombre_causa` | |
| `year` | `anio` | → INT |
| `sex` | `sexo` | `'Female'`→`'F'`, `'Male'`→`'M'`, `'All'`→`'Ambos'` |
| `age_group_code` | `grupo_etario` | `'Age25_29'`→`'25-29'`, `'Age_all'`→`'Todas'`, etc. |
| `age_group` | — eliminada | Formato `[25-29]` redundante con `grupo_etario` |
| `number` | `defunciones` | → BIGINT |
| `percentage_of_cause_specific_deaths_out_of_total_deaths` | `pct_causa_total_muertes` | NULL preservado |
| `age_standardized_death_rate_per_100_000_standard_population` | `tasa_estandarizada_x100k` | NULL preservado |
| `death_rate_per_100_000_population` | `tasa_mortalidad_x100k` | NULL preservado |
| `country` | `pais` | |
| `volume_path` / `gdrive_file_id` | — eliminadas | Linaje Bronze-específico; `source_file` ya lo captura |

In [ ]:
# ── Setup: imports, config y helpers en una sola celda ────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

BRONZE_SCHEMA = "sandbox_bronze_ss2"
SILVER_SCHEMA = "stage_silver_ss2"
WRITE_MODE    = "overwrite"

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return "manual"


def normalize_sex(col_expr):
    """
    'Female' → 'F'  |  'Male' → 'M'  |  'All' → 'Ambos'
    Cualquier otro valor se conserva sin cambios.
    """
    return (
        F.when(col_expr == "Female", F.lit("F"))
         .when(col_expr == "Male",   F.lit("M"))
         .when(col_expr == "All",    F.lit("Ambos"))
         .otherwise(col_expr)
    )


def normalize_age_group(col_expr):
    """
    'Age_all'     → 'Todas'
    'Age_unknown' → 'Desconocido'
    'Age85_over'  → '85+'
    'Age00'       → '0'
    'Age25_29'    → '25-29'  (patrón general: strip Age, _ → -)
    """
    return (
        F.when(col_expr == "Age_all",     F.lit("Todas"))
         .when(col_expr == "Age_unknown", F.lit("Desconocido"))
         .when(col_expr == "Age85_over",  F.lit("85+"))
         .when(col_expr == "Age00",       F.lit("0"))
         .otherwise(
             F.regexp_replace(
                 F.regexp_replace(col_expr, r'^Age', ''),
                 r'_', '-'
             )
         )
    )


def drop_if_exists(df, *col_names):
    """Elimina columnas solo si existen — maneja diferencias entre tablas."""
    for col_name in col_names:
        if col_name in df.columns:
            df = df.drop(col_name)
    return df


def add_silver_audit(df, run_id):
    return (
        df
        .withColumn("silver_processed_timestamp", F.current_timestamp())
        .withColumn("silver_job_run_id",           F.lit(run_id))
    )




# ── Catálogo OMS: código → nombre en español ──────────────────────────────────
_OMS_INDICATOR_ROWS = [
    ("CG0000", "Todas las causas"),
    ("CG0010", "Enfermedades transmisibles, maternas, perinatales y nutricionales"),
    ("CG0020", "Enfermedades infecciosas y parasitarias"),
    ("CG0030", "Tuberculosis"),
    ("CG0040", "Enfermedades de transmisión sexual excluido VIH"),
    ("CG0050", "Sífilis"),
    ("CG0060", "Clamidia"),
    ("CG0070", "Gonorrea"),
    ("CG0080", "Otras enfermedades de transmisión sexual"),
    ("CG0090", "VIH/SIDA"),
    ("CG0100", "Enfermedades diarreicas"),
    ("CG0110", "Enfermedades prevenibles por vacunación en la infancia"),
    ("CG0120", "Tos ferina (pertussis)"),
    ("CG0130", "Poliomielitis"),
    ("CG0140", "Difteria"),
    ("CG0150", "Sarampión"),
    ("CG0160", "Tétanos"),
    ("CG0170", "Meningitis"),
    ("CG0180", "Hepatitis B"),
    ("CG0190", "Hepatitis C"),
    ("CG0200", "Malaria (paludismo)"),
    ("CG0210", "Enfermedades tropicales agrupadas"),
    ("CG0220", "Tripanosomiasis"),
    ("CG0230", "Enfermedad de Chagas"),
    ("CG0240", "Esquistosomiasis"),
    ("CG0250", "Leishmaniasis"),
    ("CG0260", "Filariasis linfática"),
    ("CG0270", "Oncocercosis"),
    ("CG0280", "Lepra"),
    ("CG0290", "Dengue"),
    ("CG0300", "Encefalitis japonesa"),
    ("CG0310", "Tracoma"),
    ("CG0320", "Infecciones intestinales por nematodos"),
    ("CG0330", "Ascariasis"),
    ("CG0340", "Tricuriasis"),
    ("CG0350", "Anquilostomiasis"),
    ("CG0360", "Otras infecciones intestinales"),
    ("CG0370", "Otras enfermedades infecciosas"),
    ("CG0380", "Infecciones respiratorias"),
    ("CG0390", "Infecciones respiratorias bajas"),
    ("CG0395", "COVID-19"),
    ("CG0400", "Infecciones respiratorias altas"),
    ("CG0410", "Otitis media"),
    ("CG0420", "Afecciones maternas"),
    ("CG0430", "Hemorragia materna"),
    ("CG0440", "Sepsis materna"),
    ("CG0450", "Trastornos hipertensivos del embarazo"),
    ("CG0460", "Parto obstruido"),
    ("CG0470", "Aborto"),
    ("CG0480", "Otras afecciones maternas"),
    ("CG0490", "Afecciones perinatales"),
    ("CG0500", "Bajo peso al nacer"),
    ("CG0510", "Asfixia y traumatismo al nacer"),
    ("CG0520", "Otras afecciones perinatales"),
    ("CG0530", "Deficiencias nutricionales"),
    ("CG0540", "Desnutrición proteico-energética"),
    ("CG0550", "Deficiencia de yodo"),
    ("CG0560", "Deficiencia de vitamina A"),
    ("CG0570", "Anemia por deficiencia de hierro"),
    ("CG0580", "Otros trastornos nutricionales"),
    ("CG0590", "Enfermedades no transmisibles"),
    ("CG0600", "Neoplasias malignas"),
    ("CG0610", "Cánceres de boca y orofaringe"),
    ("CG0611", "a. Cáncer de labio y cavidad oral"),
    ("CG0612", "b. Cáncer de nasofaringe"),
    ("CG0613", "c. Cáncer de otras partes de la faringe"),
    ("CG0620", "Cáncer de esófago"),
    ("CG0630", "Cáncer de estómago"),
    ("CG0640", "Cáncer de colon y recto"),
    ("CG0650", "Cáncer de hígado"),
    ("CG0660", "Cáncer de páncreas"),
    ("CG0670", "Cáncer de tráquea, bronquios y pulmón"),
    ("CG0680", "Melanoma y otros cánceres de piel"),
    ("CG0681", "a. Melanoma maligno de piel"),
    ("CG0682", "b. Cáncer de piel no melanoma"),
    ("CG0690", "Cáncer de mama"),
    ("CG0700", "Cáncer de cuello uterino"),
    ("CG0710", "Cáncer de cuerpo uterino"),
    ("CG0720", "Cáncer de ovario"),
    ("CG0730", "Cáncer de próstata"),
    ("CG0731", "Cáncer testicular"),
    ("CG0732", "Cáncer de riñón"),
    ("CG0740", "Cáncer de vejiga"),
    ("CG0741", "Cánceres del encéfalo y sistema nervioso"),
    ("CG0742", "Cáncer de vesícula biliar y vías biliares"),
    ("CG0743", "Cáncer de laringe"),
    ("CG0744", "Cáncer de tiroides"),
    ("CG0745", "Mesotelioma"),
    ("CG0750", "Linfomas y mieloma múltiple"),
    ("CG0751", "a. Linfoma de Hodgkin"),
    ("CG0752", "b. Linfoma no Hodgkin"),
    ("CG0753", "c. Mieloma múltiple"),
    ("CG0760", "Leucemia"),
    ("CG0768", "Neoplasias malignas mal definidas"),
    ("CG0769", "Otras neoplasias malignas"),
    ("CG0780", "Otras neoplasias"),
    ("CG0789", "Diabetes mellitus y trastornos endocrinos"),
    ("CG0790", "Diabetes mellitus"),
    ("CG0800", "Trastornos endocrinos"),
    ("CG0810", "Trastornos neuropsiquiátricos"),
    ("CG0820", "Trastornos depresivos unipolares"),
    ("CG0830", "Trastorno bipolar"),
    ("CG0840", "Esquizofrenia"),
    ("CG0850", "Epilepsia"),
    ("CG0860", "Trastornos por consumo de alcohol"),
    ("CG0870", "Enfermedad de Alzheimer y otras demencias"),
    ("CG0880", "Enfermedad de Parkinson"),
    ("CG0890", "Esclerosis múltiple"),
    ("CG0900", "Trastornos por consumo de drogas"),
    ("CG0910", "Trastorno de estrés postraumático"),
    ("CG0920", "Trastorno obsesivo-compulsivo"),
    ("CG0930", "Trastorno de pánico"),
    ("CG0940", "Insomnio (primario)"),
    ("CG0950", "Migraña"),
    ("CG0960", "Discapacidad intelectual"),
    ("CG0970", "Otros trastornos neuropsiquiátricos"),
    ("CG0980", "Enfermedades de los órganos de los sentidos"),
    ("CG0990", "Glaucoma"),
    ("CG1000", "Cataratas"),
    ("CG1010", "Trastornos visuales relacionados con la edad"),
    ("CG1020", "Pérdida auditiva de inicio en la edad adulta"),
    ("CG1030", "Otros trastornos de los órganos de los sentidos"),
    ("CG1040", "Enfermedades cardiovasculares"),
    ("CG1050", "Cardiopatía reumática"),
    ("CG1060", "Cardiopatía hipertensiva"),
    ("CG1070", "Cardiopatía isquémica"),
    ("CG1080", "Enfermedad cerebrovascular"),
    ("CG1090", "Enfermedades inflamatorias del corazón"),
    ("CG1098", "Enfermedades circulatorias mal definidas"),
    ("CG1099", "Otras enfermedades circulatorias"),
    ("CG1110", "Enfermedades respiratorias"),
    ("CG1120", "Enfermedad pulmonar obstructiva crónica (EPOC)"),
    ("CG1130", "Asma"),
    ("CG1140", "Otras enfermedades respiratorias"),
    ("CG1150", "Enfermedades digestivas"),
    ("CG1160", "Úlcera péptica"),
    ("CG1170", "Cirrosis hepática"),
    ("CG1180", "Apendicitis"),
    ("CG1181", "Gastritis y duodenitis"),
    ("CG1182", "Íleo paralítico y obstrucción intestinal"),
    ("CG1183", "Enfermedad inflamatoria intestinal"),
    ("CG1184", "Enfermedades de la vesícula biliar y vías biliares"),
    ("CG1185", "Pancreatitis"),
    ("CG1190", "Otras enfermedades digestivas"),
    ("CG1200", "Enfermedades genitourinarias"),
    ("CG1210", "Nefritis y nefrosis"),
    ("CG1220", "Hipertrofia benigna de próstata"),
    ("CG1230", "Otras enfermedades del sistema genitourinario"),
    ("CG1240", "Enfermedades de la piel"),
    ("CG1250", "Enfermedades musculoesqueléticas"),
    ("CG1260", "Artritis reumatoide"),
    ("CG1270", "Osteoartritis"),
    ("CG1280", "Gota"),
    ("CG1290", "Dolor de espalda"),
    ("CG1300", "Otros trastornos musculoesqueléticos"),
    ("CG1310", "Anomalías congénitas"),
    ("CG1320", "Defecto de la pared abdominal"),
    ("CG1330", "Anencefalia"),
    ("CG1340", "Atresia anorrectal"),
    ("CG1350", "Labio leporino"),
    ("CG1360", "Paladar hendido"),
    ("CG1370", "Atresia esofágica"),
    ("CG1380", "Agenesia renal"),
    ("CG1390", "Síndrome de Down"),
    ("CG1400", "Anomalías congénitas del corazón"),
    ("CG1410", "Espina bífida"),
    ("CG1420", "Otras anomalías congénitas"),
    ("CG1430", "Afecciones bucales"),
    ("CG1440", "Caries dental"),
    ("CG1450", "Enfermedad periodontal"),
    ("CG1460", "Edentulismo"),
    ("CG1470", "Otras enfermedades bucales"),
    ("CG1475", "Síndrome de muerte súbita del lactante"),
    ("CG1480", "Traumatismos"),
    ("CG1490", "Traumatismos no intencionales"),
    ("CG1500", "Accidentes de tráfico"),
    ("CG1510", "Intoxicaciones"),
    ("CG1520", "Caídas"),
    ("CG1530", "Incendios"),
    ("CG1540", "Ahogamiento"),
    ("CG1541", "Exposición a fuerzas mecánicas"),
    ("CG1542", "Desastres naturales"),
    ("CG1550", "Otros traumatismos no intencionales"),
    ("CG1560", "Traumatismos intencionales"),
    ("CG1570", "Lesiones autoinfligidas"),
    ("CG1580", "Violencia"),
    ("CG1590", "Guerra"),
    ("CG1600", "Otras lesiones intencionales"),
    ("CG1605", "Traumatismos e incidentes mal definidos"),
    ("CG1610", "Enfermedades mal definidas")
]

ref_oms_df = spark.createDataFrame(
    _OMS_INDICATOR_ROWS,
    ["codigo_oms", "nombre_espanol"]
)


def translate_nombre_causa(df, ref_oms_df):
    """Traduce nombre_causa al español via JOIN con catálogo OMS por codigo_causa."""
    return (
        df
        .join(F.broadcast(ref_oms_df), F.col("codigo_causa") == F.col("codigo_oms"), "left")
        .withColumn("nombre_causa", F.coalesce(F.col("nombre_espanol"), F.col("nombre_causa")))
        .drop("codigo_oms", "nombre_espanol")
    )

RUN_ID = get_job_run_id()
logger.info(f"Setup OK — run_id={RUN_ID}")

## Tabla 1: who_guatemala
Fuente Bronze: Databricks Volume (`/Volumes/ss2_dw_workspace/sandbox_bronze_ss2/oms_local/`)  
Columna de linaje Bronze: `volume_path` — se elimina en Silver.

In [ ]:
TABLE      = "who_guatemala"
bronze_tbl = f"{BRONZE_SCHEMA}.{TABLE}"
silver_tbl = f"{SILVER_SCHEMA}.{TABLE}"

df = spark.table(bronze_tbl)
logger.info(f"[{TABLE}] Bronze: {df.count():,} filas — columnas: {df.columns}")

df_silver = (
    df
    # ── Renombrar: inglés críptico → español descriptivo ─────────────────────
    .withColumnRenamed("indicator_code", "codigo_causa")
    .withColumnRenamed("indicator_name", "nombre_causa")
    .withColumnRenamed("number",         "defunciones")
    .withColumnRenamed("country",        "pais")
    .withColumnRenamed(
        "percentage_of_cause_specific_deaths_out_of_total_deaths",
        "pct_causa_total_muertes"
    )
    .withColumnRenamed(
        "age_standardized_death_rate_per_100_000_standard_population",
        "tasa_estandarizada_x100k"
    )
    .withColumnRenamed(
        "death_rate_per_100_000_population",
        "tasa_mortalidad_x100k"
    )
    # ── Cast tipos ────────────────────────────────────────────────────────────
    .withColumnRenamed("year", "anio")
    .withColumn("anio",        F.col("anio").cast(IntegerType()))
    .withColumn("defunciones", F.col("defunciones").cast(LongType()))
    # ── Normalizar sexo y grupo etario ────────────────────────────────────────
    .withColumn("sexo",         normalize_sex(F.col("sex")))
    .withColumn("grupo_etario", normalize_age_group(F.col("age_group_code")))
    .drop("sex", "age_group_code")
    # age_group Bronze = "[25-29]" bracket format: redundante con grupo_etario
    .drop("age_group")
    # Tasas: NULL = no calculable para ese grupo etario — no imputar 0
    .dropDuplicates()
)

df_silver = translate_nombre_causa(df_silver, ref_oms_df)
df_silver = drop_if_exists(df_silver, "volume_path")
df_silver = add_silver_audit(df_silver, RUN_ID)

logger.info(f"[{TABLE}] Silver: {df_silver.count():,} filas")
(
    df_silver.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(silver_tbl)
)
logger.info(f"Escrito → {silver_tbl}")

## Tabla 2: who_costa_rica
Fuente Bronze: Google Drive API v3  
Columna de linaje Bronze: `gdrive_file_id` — se elimina en Silver.

In [ ]:
TABLE      = "who_costa_rica"
bronze_tbl = f"{BRONZE_SCHEMA}.{TABLE}"
silver_tbl = f"{SILVER_SCHEMA}.{TABLE}"

df = spark.table(bronze_tbl)
logger.info(f"[{TABLE}] Bronze: {df.count():,} filas — columnas: {df.columns}")

df_silver = (
    df
    # ── Renombrar: inglés críptico → español descriptivo ─────────────────────
    .withColumnRenamed("indicator_code", "codigo_causa")
    .withColumnRenamed("indicator_name", "nombre_causa")
    .withColumnRenamed("number",         "defunciones")
    .withColumnRenamed("country",        "pais")
    .withColumnRenamed(
        "percentage_of_cause_specific_deaths_out_of_total_deaths",
        "pct_causa_total_muertes"
    )
    .withColumnRenamed(
        "age_standardized_death_rate_per_100_000_standard_population",
        "tasa_estandarizada_x100k"
    )
    .withColumnRenamed(
        "death_rate_per_100_000_population",
        "tasa_mortalidad_x100k"
    )
    # ── Cast tipos ────────────────────────────────────────────────────────────
    .withColumnRenamed("year", "anio")
    .withColumn("anio",        F.col("anio").cast(IntegerType()))
    .withColumn("defunciones", F.col("defunciones").cast(LongType()))
    # ── Normalizar sexo y grupo etario ────────────────────────────────────────
    .withColumn("sexo",         normalize_sex(F.col("sex")))
    .withColumn("grupo_etario", normalize_age_group(F.col("age_group_code")))
    .drop("sex", "age_group_code")
    .drop("age_group")
    .dropDuplicates()
)

df_silver = translate_nombre_causa(df_silver, ref_oms_df)
df_silver = drop_if_exists(df_silver, "gdrive_file_id")
df_silver = add_silver_audit(df_silver, RUN_ID)

logger.info(f"[{TABLE}] Silver: {df_silver.count():,} filas")
(
    df_silver.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(silver_tbl)
)
logger.info(f"Escrito → {silver_tbl}")

## Validación — Row counts y calidad

In [ ]:
_TABLES = ["who_guatemala", "who_costa_rica"]

print(f"{'Tabla':<20} {'Bronze':>10} {'Silver':>10} {'Estado':>8}")
print("-" * 52)
for t in _TABLES:
    b = spark.table(f"{BRONZE_SCHEMA}.{t}").count()
    s = spark.table(f"{SILVER_SCHEMA}.{t}").count()
    estado = "OK" if b == s else f"DIFF ({b - s:+,})"
    print(f"{t:<20} {b:>10,} {s:>10,} {estado:>8}")

print("\n── Columnas Bronze eliminadas (no deben aparecer en Silver) ──")
_BRONZE_COLS = {"indicator_code", "indicator_name", "number", "year", "sex",
                "age_group_code", "age_group", "country",
                "percentage_of_cause_specific_deaths_out_of_total_deaths",
                "age_standardized_death_rate_per_100_000_standard_population",
                "death_rate_per_100_000_population",
                "volume_path", "gdrive_file_id"}
for t in _TABLES:
    cols = set(spark.table(f"{SILVER_SCHEMA}.{t}").columns)
    leaked = cols & _BRONZE_COLS
    print(f"  {t}: {'OK' if not leaked else f'WARN columnas Bronze presentes: {leaked}'}")

print("\n── Valores distintos de sexo (debe ser solo F, M, Ambos) ──")
for t in _TABLES:
    vals = spark.table(f"{SILVER_SCHEMA}.{t}").select("sexo").distinct().collect()
    print(f"  {t}: {sorted([r.sexo for r in vals])}")

print("\n── grupo_etario normalizado — sin prefijo Age ni guión bajo ──")
(
    spark.table(f"{SILVER_SCHEMA}.who_guatemala")
    .select("grupo_etario").distinct()
    .orderBy("grupo_etario")
    .show(30, truncate=False)
)

# ── Diagnóstico NULLs en tasas ─────────────────────────────────────────────────
# La tasa estandarizada por edad (age_standardized_death_rate) es un agregado que WHO
# solo calcula para la fila "todos los grupos etarios" (grupo_etario='Todas').
# Las filas por grupo individual (0-4, 5-9, ...) la traen NULL en el CSV fuente.
# Si Bronze y Silver tienen el mismo número de NULLs → origen en fuente, no en nuestra transformación.
print("\n── Diagnóstico NULLs: Bronze vs Silver (¿los introducimos nosotros?) ──")
_BRONZE_EST_COL = "age_standardized_death_rate_per_100_000_standard_population"
for t in _TABLES:
    b_total = spark.table(f"{BRONZE_SCHEMA}.{t}").count()
    b_nulls = spark.table(f"{BRONZE_SCHEMA}.{t}").filter(F.col(_BRONZE_EST_COL).isNull()).count()
    s_total = spark.table(f"{SILVER_SCHEMA}.{t}").count()
    s_nulls = spark.table(f"{SILVER_SCHEMA}.{t}").filter(F.col("tasa_estandarizada_x100k").isNull()).count()
    match = "OK — NULLs idénticos, origen en fuente" if b_nulls == s_nulls else f"WARN — diferencia {s_nulls - b_nulls:+,}"
    print(f"\n  {t}:")
    print(f"    Bronze : {b_nulls:,}/{b_total:,} NULLs ({100*b_nulls/b_total:.1f}%)")
    print(f"    Silver : {s_nulls:,}/{s_total:,} NULLs ({100*s_nulls/s_total:.1f}%)")
    print(f"    Veredicto: {match}")

print("\n── Distribución por grupo_etario: ¿solo 'Todas' tiene tasa_estandarizada_x100k? ──")
print("  Esperado: con_valor > 0 SOLO para grupo_etario = 'Todas'; resto nulos.\n")
for t in _TABLES:
    print(f"  {t}:")
    (
        spark.table(f"{SILVER_SCHEMA}.{t}")
        .groupBy("grupo_etario")
        .agg(
            F.count("*").alias("total_filas"),
            F.count("tasa_estandarizada_x100k").alias("con_valor"),
            F.sum(F.col("tasa_estandarizada_x100k").isNull().cast("int")).alias("nulos")
        )
        .orderBy("grupo_etario")
        .show(30, truncate=False)
    )